# 02 · Feature maps

In notebook 01 we saw that classic methods only look at brightness or color. A neural network, on the other hand, can recognize a dog. But what does it actually see when it looks at an image?

It all rests on a simple idea. A **kernel** (or filter) is a small grid of numbers. When we slide it over an image, it produces a **feature map**. And each kernel highlights a different pattern: vertical edges, horizontal edges, colors, textures.

We'll check this step by step:
1. An image is just numbers
2. Convolution by hand, on a tiny image
3. Kernels on a real photo
4. Why a kernel is a pattern detector
5. Color images
6. Several kernels = one convolution layer (`Conv2d`)
7. Stride, padding and activation
8. The kernels learned by a real network (YOLO)
9. What happens deep inside the network

**Prerequisites:** notebook 01. No deep learning background needed, you just need to know how to multiply and add.

## 0. Setup

The imports, and a small `show()` function to display several images side by side.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams["figure.dpi"] = 100


def show(images, titles=None, cmap="gray", size=3.2, colorbar=False):
    # Display a list of images side by side
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(size * n, size))
    axes = np.atleast_1d(axes)
    for i, (ax, im) in enumerate(zip(axes, images)):
        h = ax.imshow(im, cmap=cmap)
        if titles:
            ax.set_title(titles[i], fontsize=10)
        ax.axis("off")
        if colorbar:
            fig.colorbar(h, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()

## 1. An image is just numbers

For a computer, an image is an array of numbers:
- in grayscale, one number per pixel, from 0 (black) to 255 (white);
- in color, three numbers per pixel (red, green, blue).

In [ ]:
img_bgr = cv2.imread("../../img/dog_cat.png")              # OpenCV reads in BGR (Blue, Green, Red)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)   # matplotlib expects RGB
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

print("Color image:", img_rgb.shape, "-> (height, width, 3 channels)")
print("Gray image: ", gray.shape, "-> (height, width)")

show([img_rgb, gray], ["Color", "Grayscale"], size=5)

Let's zoom into an 8×8 pixel square to see the numbers behind the image. We pick a spot where the black dog's ear stands out against the light background.

In [ ]:
y, x = 20, 110   # top-left corner of the square (row, column)
patch = gray[y:y + 8, x:x + 8]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(gray, cmap="gray")
axes[0].add_patch(plt.Rectangle((x, y), 8, 8, edgecolor="red", facecolor="none", lw=2))
axes[0].set_title("Where we zoom (red square)")
axes[0].axis("off")

axes[1].imshow(patch, cmap="gray", vmin=0, vmax=255)
for (i, j), v in np.ndenumerate(patch):
    axes[1].text(j, i, v, ha="center", va="center", fontsize=9,
                 color="white" if v < 128 else "black")
axes[1].set_title("The zoom: each cell is a number")
axes[1].axis("off")
plt.show()

Everything we do next is just arithmetic on these numbers.

## 2. Convolution by hand

### 2.1 A tiny image

We build a 6×6 pixel image: the left half is dark (0), the right half is bright (10). So there is a vertical edge in the middle.

In [ ]:
tiny = np.array([
    [0, 0, 0, 10, 10, 10],
    [0, 0, 0, 10, 10, 10],
    [0, 0, 0, 10, 10, 10],
    [0, 0, 0, 10, 10, 10],
    [0, 0, 0, 10, 10, 10],
    [0, 0, 0, 10, 10, 10],
], dtype=np.float32)


def show_grid(ax, M, title, cmap="gray"):
    # Display a matrix with its values written in each cell
    lim = np.abs(M).max() or 1
    if cmap == "bwr":
        ax.imshow(M, cmap=cmap, vmin=-lim, vmax=lim)
    else:
        ax.imshow(M, cmap=cmap)
    for (i, j), v in np.ndenumerate(M):
        ax.text(j, i, f"{v:g}", ha="center", va="center", fontsize=11,
                color="red" if cmap == "gray" else "black")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])


fig, ax = plt.subplots(figsize=(4, 4))
show_grid(ax, tiny, "6×6 image with a vertical edge")
plt.show()

### 2.2 The kernel

Here is a well-known 3×3 kernel, the horizontal Sobel filter (Sobel X):

$$
K = \begin{pmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{pmatrix}
$$

The left column is negative, so we subtract the pixels on the left. The right column is positive, so we add the ones on the right. The middle column is 0, we ignore it.

In other words, this kernel computes "right minus left".

In [ ]:
sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=np.float32)

fig, ax = plt.subplots(figsize=(3, 3))
show_grid(ax, sobel_x, "Sobel X kernel", cmap="bwr")
plt.show()

### 2.3 One single computation, step by step

A convolution is three moves:
1. place the kernel on a 3×3 piece of the image;
2. multiply each kernel cell by the image cell underneath;
3. add up the 9 results.

You get a single number, which becomes one pixel of the output map. Let's try it in two places: right in the dark area, then on the edge.

In [ ]:
def one_step(image, kernel, row, col):
    piece = image[row:row + 3, col:col + 3]
    product = piece * kernel
    total = product.sum()

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))
    show_grid(axes[0], piece, f"Piece of the image\n(row {row}, column {col})")
    show_grid(axes[1], kernel, "× Kernel", cmap="bwr")
    show_grid(axes[2], product, f"= Cell-by-cell product\nSum = {total:g}", cmap="bwr")
    plt.tight_layout()
    plt.show()
    return total


one_step(tiny, sobel_x, row=0, col=0)   # uniform area
one_step(tiny, sobel_x, row=0, col=1)   # straddling the edge

In the uniform area, left equals right and the sum is 0: the kernel sees nothing. On the edge, the right is brighter than the left and the sum is 40: the kernel reacts strongly.

### 2.4 Sliding the kernel everywhere

We repeat this computation at every position in the image, left to right and top to bottom. The function below does it with two `for` loops. That's all there is to convolution.

In [ ]:
def convolution(image, kernel):
    # 2D convolution by hand (no padding, stride 1)
    kh, kw = kernel.shape
    H, W = image.shape
    output = np.zeros((H - kh + 1, W - kw + 1), dtype=np.float32)
    for i in range(output.shape[0]):          # for each row
        for j in range(output.shape[1]):      # for each column
            piece = image[i:i + kh, j:j + kw]
            output[i, j] = (piece * kernel).sum()
    return output


fmap = convolution(tiny, sobel_x)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
show_grid(axes[0], tiny, "6×6 image")
show_grid(axes[1], sobel_x, "Kernel", cmap="bwr")
show_grid(axes[2], fmap, "4×4 feature map", cmap="magma")
plt.tight_layout()
plt.show()

That's our first feature map. It equals 40 exactly where the vertical edge is, and 0 everywhere else. So Sobel X is a vertical edge detector.

Why is the output 4×4 and not 6×6? The 3×3 kernel can't stick out of the image, so there are only 4 possible positions per row (6 − 3 + 1 = 4). *Padding*, which we'll see later, fixes this.

What happens if we rotate the image a quarter turn, so the edge becomes horizontal?

In [ ]:
tiny_horizontal = tiny.T   # transpose: the vertical edge becomes horizontal

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.8))
show_grid(axes[0], tiny_horizontal, "Horizontal edge")
show_grid(axes[1], convolution(tiny_horizontal, sobel_x), "Sobel X output", cmap="magma")
plt.tight_layout()
plt.show()

The map is entirely zero: Sobel X doesn't see horizontal edges. To see them, we need another kernel. That's the starting idea: one kernel, one type of pattern.

## 3. Kernels on a real photo

Now we apply our `convolution` function to the grayscale photo, and compare it with OpenCV's `cv2.filter2D`, which does the same computation much faster.

In [ ]:
import time

g = gray.astype(np.float32) / 255.0   # bring pixels between 0 and 1

t0 = time.time()
map_by_hand = convolution(g, sobel_x)
t1 = time.time()
map_cv = cv2.filter2D(g, cv2.CV_32F, sobel_x)[1:-1, 1:-1]   # drop the borders to get the same size
t2 = time.time()

print(f"Our function: {t1 - t0:.3f} s")
print(f"OpenCV:       {t2 - t1:.4f} s")
print("Same results?", np.allclose(map_by_hand, map_cv, atol=1e-5))

show([g, np.abs(map_by_hand)], ["Image", "|Sobel X|: vertical edges"], cmap="magma", size=5)

Both results are identical. OpenCV and PyTorch do exactly this computation, just more optimized.

We display the absolute value because a "dark → bright" edge gives a positive number, and a "bright → dark" edge a negative one. Either way, it's an edge.

### 3.1 A few classic kernels

For each kernel, look at its numbers first, then at what lights up in the map.

In [ ]:
KERNELS = {
    "Sobel X\n(vertical edges)":     np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "Sobel Y\n(horizontal edges)":   np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
    "Diagonal\n(45° edges)":         np.array([[0, 1, 2], [-1, 0, 1], [-2, -1, 0]]),
    "Laplacian\n(all edges)":        np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]]),
    "Blur\n(average)":               np.ones((3, 3)) / 9,
    "Sharpen":                        np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
    "Emboss":                         np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]]),
}

fig, axes = plt.subplots(len(KERNELS), 2, figsize=(8, 2.6 * len(KERNELS)),
                         gridspec_kw={"width_ratios": [1, 2.2]})
for row, (name, k) in enumerate(KERNELS.items()):
    k = k.astype(np.float32)
    show_grid(axes[row, 0], np.round(k, 2), name, cmap="bwr")
    fmap = cv2.filter2D(g, cv2.CV_32F, k)
    # Zero-sum kernels = detectors: we show |map|. Otherwise: the filtered image.
    axes[row, 1].imshow(np.abs(fmap) if abs(k.sum()) < 1e-6 else fmap, cmap="magma")
    axes[row, 1].axis("off")
plt.tight_layout()
plt.show()

### How to read these results

| Kernel | What it computes | What lights up |
|---|---|---|
| Sobel X | right − left | vertical edges (legs, sides of the heads) |
| Sobel Y | bottom − top | horizontal edges (chin, top of the heads) |
| Diagonal | top-right − bottom-left | slanted edges |
| Laplacian | the center compared with its neighbors | all thin edges, whatever the direction |
| Blur | the average of the 9 neighbors | nothing in particular, the image is smoothed |
| Sharpen | the center boosted, the neighbors removed | a more contrasted image |
| Emboss | one side − the opposite side | a 3D effect, like an engraving |

A simple rule: when a kernel's values add up to 0 (Sobel, Laplacian…), it's a detector. It gives 0 on flat areas and only reacts to changes.

## 4. Why a kernel is a pattern detector

The sum of products is largest when the image piece looks like the kernel: bright pixels land on positive weights, dark pixels on negative weights.

To check this, we find where the Sobel X map is strongest and look at the image piece at that spot.

In [ ]:
fmap = cv2.filter2D(g, cv2.CV_32F, sobel_x)
i, j = np.unravel_index(np.argmax(fmap), fmap.shape)   # position of the maximum value
piece = g[i - 1:i + 2, j - 1:j + 2]

print(f"Maximum response = {fmap[i, j]:.2f} at position (row={i}, column={j})")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].imshow(g, cmap="gray")
axes[0].plot(j, i, "r+", markersize=20, mew=3)
axes[0].set_title("Where Sobel X reacts the most")
axes[0].axis("off")
show_grid(axes[1], np.round(piece, 2), "The image piece at that spot")
show_grid(axes[2], sobel_x, "The Sobel X kernel", cmap="bwr")
plt.tight_layout()
plt.show()

The piece is dark on the left and bright on the right. It has the same shape as the kernel, negative on the left and positive on the right.

So at every point, the map answers the question: "does the image here look like my kernel?" That's why it's called a *feature* map: it shows where a specific pattern is in the image.

## 5. What about color images?

A color image has 3 channels (R, G, B). So a color kernel has 3 layers: it's a 3×3×3 cube.

The computation doesn't change. Each kernel layer is applied to its channel (R with R, G with G, B with B), then we add up the 3 results. We get a single map.

Let's build a kernel that detects ginger and orange: +1 on red, −0.5 on green, −1 on blue.

In [ ]:
rgb = img_rgb.astype(np.float32) / 255.0

# 3×3×3 kernel: one 3×3 grid per channel. We only put a weight in the center.
ginger_kernel = np.zeros((3, 3, 3), dtype=np.float32)
ginger_kernel[1, 1, 0] = 1.0     # Red channel: add
ginger_kernel[1, 1, 1] = -0.5    # Green channel: subtract a little
ginger_kernel[1, 1, 2] = -1.0    # Blue channel: subtract

# Step 1: one computation per channel
maps_per_channel = [cv2.filter2D(rgb[:, :, c], cv2.CV_32F, ginger_kernel[:, :, c]) for c in range(3)]
# Step 2: add up the 3 maps
ginger_map = sum(maps_per_channel)

show([rgb, *maps_per_channel, ginger_map],
     ["Image", "Red × (+1)", "Green × (−0.5)", "Blue × (−1)", "SUM = \"ginger\" map"],
     cmap="magma", size=3.3)

The ginger dog on the left and the tabby cat light up, while the white dog and the black dog stay dark.

So a kernel can detect a shape (edges), a color, or both at once. We'll find this kind of kernel inside YOLO.

## 6. Several kernels = one convolution layer

In a neural network, a convolution layer (`nn.Conv2d` in PyTorch) is nothing more than a bunch of kernels. Each kernel produces its map, and all the maps are stacked. A layer with 4 kernels produces 4 maps: we say the output has 4 channels.

To show this, we create an `nn.Conv2d` layer, put our own kernels in it (Sobel X, Sobel Y, Laplacian, Blur), and look at what comes out.

In [ ]:
my_kernels = [
    np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),     # Sobel X
    np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),     # Sobel Y
    np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]]),       # Laplacian
    np.ones((3, 3)) / 9,                                # Blur
]

# 1 input channel (grayscale image) -> 4 output channels (4 kernels)
layer = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1, bias=False)

# Conv2d weights have shape (n_kernels, n_input_channels, height, width)
print("Weight shape:", tuple(layer.weight.shape))

with torch.no_grad():
    layer.weight[:] = torch.tensor(np.stack(my_kernels)[:, None], dtype=torch.float32)

x = torch.tensor(g)[None, None]   # shape (batch=1, channels=1, H, W)
with torch.no_grad():
    y = layer(x)

print("Input:", tuple(x.shape), " -> Output:", tuple(y.shape))
show([y[0, c].abs().numpy() for c in range(4)],
     ["Channel 0: Sobel X", "Channel 1: Sobel Y", "Channel 2: Laplacian", "Channel 3: Blur"],
     cmap="magma", size=3.6)

This is the central idea of this notebook:

$$
\text{1 Conv2d layer} = N \text{ kernels} \;\longrightarrow\; N \text{ stacked feature maps}
$$

The only difference with a real network is that here we chose the kernel numbers ourselves. In a trained network, these numbers are learned automatically, to detect what is useful for the task, for example finding dogs and cats.

## 7. Stride, padding and activation

### 7.1 Padding

We add a border of zeros around the image so the kernel can also sit on the edges. With `padding=1` and a 3×3 kernel, the output keeps the same size as the input.

### 7.2 Stride

It's how many pixels we shift the kernel each time. With `stride=2`, we skip every other pixel, and the output is 2 times smaller in height and width.

The output size is given by this formula:

$$
\text{output} = \left\lfloor \frac{\text{input} + 2 \times \text{padding} - \text{kernel size}}{\text{stride}} \right\rfloor + 1
$$

In [ ]:
H, W = g.shape
print(f"Input image: {H} × {W}\n")

for padding, stride in [(0, 1), (1, 1), (1, 2), (1, 4)]:
    c = nn.Conv2d(1, 1, kernel_size=3, padding=padding, stride=stride)
    out_size = c(x).shape[-2:]
    formula = ((H + 2 * padding - 3) // stride + 1, (W + 2 * padding - 3) // stride + 1)
    print(f"padding={padding}, stride={stride}  ->  output {tuple(out_size)}   (formula: {formula})")

In [ ]:
stride_maps = []
for s in [1, 2, 4]:
    with torch.no_grad():
        out = F.conv2d(x, layer.weight[:1], stride=s, padding=1)
    stride_maps.append(out[0, 0].abs().numpy())

show(stride_maps,
     [f"stride={s} -> {m.shape[0]}×{m.shape[1]}" for s, m in zip([1, 2, 4], stride_maps)],
     cmap="magma", size=4)

The bigger the stride, the smaller and coarser the map. We lose detail, but the next computations go much faster, and each pixel of the map summarizes a larger area of the image (more on this in part 9).

### 7.3 Activation

After each convolution, a network applies an activation function. The simplest one is ReLU, which keeps positive values and sets negative ones to zero:

$$\text{ReLU}(v) = \max(0, v)$$

It only keeps the places where the kernel really found its pattern. YOLO uses SiLU, a smoother version, but the idea is the same.

In [ ]:
raw = cv2.filter2D(g, cv2.CV_32F, sobel_x)
relu = np.maximum(0, raw)

show([raw, relu],
     ["Raw Sobel X\n(blue = negative, red = positive)", "After ReLU\n(only dark → bright edges remain)"],
     cmap="bwr", size=5)

## 8. The kernels learned by a real network: YOLO

Let's move on to a real network, YOLO (`yolo26n-seg.pt`), an already-trained segmentation model. Nobody gave it its kernels: it learned them by looking at thousands of images.

The question: did it learn things that look like our handmade kernels?

In [ ]:
from ultralytics import YOLO

model = YOLO("../../models/yolo26n-seg.pt").model.float().eval()

# The first blocks of the network (the part that extracts features)
for i, block in enumerate(model.model[:11]):
    print(i, type(block).__name__)

The very first block is a convolution. Let's see what's inside.

In [ ]:
conv0 = model.model[0].conv
print(conv0)
print("Weight shape:", tuple(conv0.weight.shape), "-> 16 kernels of size 3 channels × 3 × 3")

16 kernels, each of size 3×3×3 (3×3 pixels over 3 colors), like our ginger detector from part 5. The `stride=(2, 2)` means the output will be 2 times smaller than the image.

We prepare the image the way YOLO expects it (RGB, values between 0 and 1, size a multiple of 32), then display each learned kernel next to the map it produces.

In [ ]:
# Prepare the image for YOLO
H0, W0 = img_rgb.shape[:2]
scale = 640 / max(H0, W0)
h32 = int(round(H0 * scale / 32)) * 32
w32 = int(round(W0 * scale / 32)) * 32
net_input = cv2.resize(img_rgb, (w32, h32)).astype(np.float32) / 255.0
net_input = torch.from_numpy(net_input).permute(2, 0, 1)[None]   # (1, 3, H, W)
print("Network input:", tuple(net_input.shape))

# Apply ONLY the 16 kernels of the first layer
weights = conv0.weight.detach()
with torch.no_grad():
    maps0 = F.conv2d(net_input, weights, stride=2, padding=1)[0]
print("Output:", tuple(maps0.shape), "-> 16 maps")

fig, axes = plt.subplots(8, 4, figsize=(12, 16))
for n in range(16):
    row, col = n // 2, (n % 2) * 2
    k = weights[n].permute(1, 2, 0).numpy()               # (3, 3, 3) as a color image
    k = (k - k.min()) / (k.max() - k.min() + 1e-8)     # rescaled between 0 and 1 to display it
    axes[row, col].imshow(k, interpolation="nearest")
    axes[row, col].set_title(f"kernel {n}", fontsize=9)
    axes[row, col + 1].imshow(maps0[n].numpy(), cmap="viridis")
    axes[row, col + 1].set_title(f"→ map {n}", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

### What we observe

Nobody wrote these kernels, and yet:
- several maps show the outlines of the animals, with a preferred direction. These are edge detectors, like Sobel: the network found on its own what humans had invented by hand;
- others react to color or brightness, the black dog or the white dogs stand out clearly. These are cousins of our ginger detector;
- a few maps are almost flat: those kernels look for a pattern that is rare in this image.

Careful, the colors of a displayed kernel are not exactly "the color it sees". We stretched its weights between 0 and 1 so we could draw it. A very colorful kernel mixes the channels unevenly, it's sensitive to color. A gray kernel treats the 3 channels the same, it's sensitive to shape.

To check the "edge detector" intuition, we compute how similar each learned kernel is to Sobel X and Sobel Y.

In [ ]:
def similarity(a, b):
    # Cosine similarity: 1 = same shape, 0 = unrelated, -1 = inverted shape
    a, b = a.flatten(), b.flatten()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


sobel_y = sobel_x.T
print("kernel | ~Sobel X | ~Sobel Y")
for n in range(16):
    k_gray = weights[n].sum(0).numpy()   # add up the 3 channels -> a "gray" 3×3 kernel
    sx, sy = similarity(k_gray, sobel_x), similarity(k_gray, sobel_y)
    mark = "  <- edge detector" if max(abs(sx), abs(sy)) > 0.7 else ""
    print(f"  {n:2d}   |  {sx:+.2f}   |  {sy:+.2f}{mark}")

A similarity close to +1 (or −1, meaning the same pattern inverted) confirms that some learned kernels are almost Sobel filters.

## 9. What happens deep inside the network

A network stacks dozens of layers. The important point: layer 2 doesn't apply its kernels to the image, but to the maps produced by layer 1. So it combines patterns that were already detected:
- layer 1: edges;
- layer 2: combined edges, which give corners, curves, textures;
- next layers: combinations of combinations, which give eyes, ears, fur;
- at the very bottom: concepts like "dog head" or "cat".

To see the maps inside the network, we use *hooks*: small functions that PyTorch calls when a layer produces its output, which let us keep a copy of it.

In [ ]:
LAYERS = [0, 1, 2, 4, 6, 8]
activations = {}


def make_hook(number):
    def hook(module, inp, out):
        activations[number] = out.detach()[0]   # keep the layer's output
    return hook


hooks = [model.model[i].register_forward_hook(make_hook(i)) for i in LAYERS]
with torch.no_grad():
    model(net_input)          # a single pass of the image through the whole network
for h in hooks:
    h.remove()                # remove the hooks once done

print("layer | type  | channels | size of each map")
for i in LAYERS:
    a = activations[i]
    print(f"  {i:3d} | {type(model.model[i]).__name__:5s} | {a.shape[0]:8d} | {a.shape[1]} × {a.shape[2]}")

In this table, the size of the maps shrinks (because of the `stride=2`) and the number of channels grows: 16, 32, 64, 128, 256. You need more detectors to represent more complex patterns, and less precision about their position.

For each layer, we display the 6 most active maps.

In [ ]:
N = 6
fig, axes = plt.subplots(len(LAYERS), N + 1, figsize=(2.5 * (N + 1), 2.3 * len(LAYERS)))
for row, i in enumerate(LAYERS):
    a = activations[i]
    axes[row, 0].imshow(img_rgb)
    axes[row, 0].set_title(f"layer {i}\n{a.shape[0]} channels, {a.shape[1]}×{a.shape[2]}", fontsize=9)
    most_active = a.abs().mean(dim=(1, 2)).argsort(descending=True)[:N]
    for col, channel in enumerate(most_active.tolist(), start=1):
        axes[row, col].imshow(a[channel].numpy(), cmap="viridis")
        axes[row, col].set_title(f"channel {channel}", fontsize=8)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

### How to read this figure

| Depth | What we see | What it means |
|---|---|---|
| Layers 0-1 | sharp outlines, color areas | simple patterns: edges, colors |
| Layer 2 | textures, background split from the animals | combined local patterns |
| Layers 4-6 | coarser blobs on the heads, the eyes | object parts |
| Layer 8 | large blurry blocks, hard to read | abstract concepts only the network understands |

The deeper we go, the less readable it is for us, but the more useful it is for the network. A cell in layer 8 no longer says "there's an edge here", more like "there's something that looks like a head in this area".

### The receptive field

Why do deep layers see bigger things? A pixel in layer 1 depends on a 3×3 square of the image. A pixel in layer 2 depends on 3×3 pixels of layer 1, each of which depends on 3×3 pixels of the image, and so on. The area seen grows with every layer, and even faster with strides.

This area is called the receptive field. We can compute it:

In [ ]:
# For a stack of 3×3 convolutions: field = field + (3 - 1) × (product of previous strides)
field, jump = 1, 1
print("layer | stride | receptive field (in image pixels)")
for number, stride in enumerate([2, 2, 1, 2, 1, 2, 1, 2]):
    field += (3 - 1) * jump
    jump *= stride
    print(f"  {number + 1:3d} | {stride:6d} | {field} × {field}")

This is a simplified computation, with a single 3×3 convolution per step. YOLO's real `C3k2` blocks contain several, so their receptive field is even larger.

After a few layers, a single pixel sees a large part of the image. That's what lets it recognize a whole object.

## Key takeaways

- An image is an array of numbers. A kernel is a small grid of numbers (3×3, or 3×3×3 for color).
- Convolution means sliding the kernel, multiplying cell by cell and adding up. One position gives one output pixel.
- The feature map tells you, at every point, whether the image looks like the kernel. Different kernel, different map.
- A `Conv2d` layer is N kernels that give N stacked maps (N channels).
- Stride reduces the size, padding keeps the borders, activation (ReLU, SiLU) keeps only positive responses.
- In a trained network, kernels are learned. The first ones look like classic filters (edges, colors).
- The deeper we go, the more layers combine the previous maps: patterns become complex and abstract, and the receptive field grows.

## Exercises

1. Guess before running: what map does the kernel `[[1,1,1],[1,1,1],[1,1,1]]` give, without dividing by 9? Why does the image turn almost white?
2. Invent a kernel that detects edges at −45° (the other diagonal). Hint: rotate the "Diagonal" kernel.
3. Change `ginger_kernel` so it detects black. Hint: negative weights on all 3 channels. Which animal lights up?
4. Redo part 8 with `dog.png`. Do the "color" kernels react to the same things?
5. Going further: in part 9, take a layer-8 map, resize it to the image size (`cv2.resize`) and overlay it on the photo (`alpha=0.5`). Which area of the image activates this channel?

**Next:** [03 · U-Net](03_unet.ipynb)